# DeepVerify EXP-001 Colab Preflight

This notebook verifies the exact environment, dataset manifest, configuration, and GPU readiness for `EXP-001-spatial-baseline` before beginning the full 140k dataset training.

In [ ]:
import os
import subprocess
import tempfile
import stat

# 1. Refuse to overwrite an existing checkout
if os.path.exists('/content/DeepVerify'):
    raise RuntimeError(
        "STOP. /content/DeepVerify already exists. "
        "Refusing to overwrite silently. Delete it manually if you intend to restart."
    )

# 2. Retrieve token exclusively from Colab Secrets — never hard-code it
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise ValueError("Empty token")
except Exception:
    raise RuntimeError(
        "STOP. GITHUB_TOKEN not found in Colab Secrets. "
        "Add your personal access token to Colab Secrets under the key GITHUB_TOKEN "
        "and enable notebook access."
    )

# 3. Create a temporary GIT_ASKPASS script that contains NO credentials.
#    The script reads the token from environment variables injected only into
#    the git subprocess — the secret is never written to the filesystem.
ASKPASS_SCRIPT = """#!/bin/sh
case "$1" in
    Username*) printf '%s\\n' "$DEEPVERIFY_GITHUB_USERNAME" ;;
    Password*) printf '%s\\n' "$DEEPVERIFY_GITHUB_TOKEN" ;;
esac
"""

askpass_fd, askpass_path = tempfile.mkstemp(prefix='dv_askpass_', suffix='.sh')
try:
    with os.fdopen(askpass_fd, 'w') as f:
        f.write(ASKPASS_SCRIPT)
    os.chmod(askpass_path, stat.S_IRUSR | stat.S_IXUSR)  # owner read+execute only

    # 4. Build a subprocess env that supplies the secret only in-memory.
    #    The normal repository URL contains NO credentials.
    env = os.environ.copy()
    env['GIT_ASKPASS'] = askpass_path
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['DEEPVERIFY_GITHUB_USERNAME'] = 'shairkhanfaizan'
    env['DEEPVERIFY_GITHUB_TOKEN'] = token

    repo_url = 'https://github.com/shairkhanfaizan/DeepVerify.git'
    result = subprocess.run(
        ['git', 'clone', repo_url, '/content/DeepVerify'],
        env=env,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Do not surface stderr — it may contain authentication details.
        raise RuntimeError(
            "STOP. git clone failed. Verify that GITHUB_TOKEN has read access to the repository."
        )
finally:
    # 5. Always remove the temporary askpass script, whether clone succeeded or not.
    if os.path.exists(askpass_path):
        os.remove(askpass_path)
    # 6. Clear secrets from the subprocess environment dict (belt-and-suspenders).
    env.pop('DEEPVERIFY_GITHUB_TOKEN', None)
    env.pop('DEEPVERIFY_GITHUB_USERNAME', None)
    env.pop('GIT_ASKPASS', None)

os.chdir('/content/DeepVerify')

# 7. Verify that the origin stored by git is exactly the plain HTTPS URL — no credentials.
EXPECTED_ORIGIN = 'https://github.com/shairkhanfaizan/DeepVerify.git'
origin_url = subprocess.check_output(
    ['git', 'config', '--get', 'remote.origin.url'], text=True
).strip()
if origin_url != EXPECTED_ORIGIN:
    raise RuntimeError(
        f"STOP. Remote origin URL is unexpected.\n"
        f"Expected: {EXPECTED_ORIGIN}\n"
        f"Got: {origin_url}"
    )

# 8. Checkout the exact approved commit.
checkout = subprocess.run(
    ['git', 'checkout', 'b193567'],
    capture_output=True, text=True
)
if checkout.returncode != 0:
    raise RuntimeError("STOP. git checkout b193567 failed.")

# 9. Verify HEAD.
head_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip()
if not head_commit.startswith('b193567'):
    raise RuntimeError(f"STOP. Expected commit b193567..., got {head_commit}")

print(f"Successfully checked out approved commit: {head_commit}")

In [ ]:
# 3. Install ML package using the approved ML environment configuration
!pip install uv
!uv pip install --system -e ml/

In [ ]:
# 4. Environment information
import sys
import torch
import torchvision

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    t = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU total memory: {t / 1024**3:.2f} GB")

In [ ]:
# 5. Require GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "STOP. CUDA MUST be available for the REAL EXP-001 experiment. "
        "Do NOT silently fall back to CPU."
    )

print("CUDA is available. Proceeding.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 6. Load EXP-001 configuration
import yaml
from pathlib import Path

config_path = Path("experiments/configs/EXP-001.yaml")
if not config_path.exists():
    raise FileNotFoundError("EXP-001 config not found!")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

def require_eq(actual, expected, name):
    if actual != expected:
        raise RuntimeError(f"STOP. Config mismatch for {name}: expected {expected}, got {actual}")

require_eq(config["experiment"]["seed"], 42, "seed")
require_eq(config["model"]["architecture"], "efficientnet-b0", "architecture")
require_eq(config["model"]["pretrained"], True, "pretrained")
require_eq(config["model"]["pretrained_weights"], "IMAGENET1K_V1", "pretrained_weights")
require_eq(config["preprocessing"]["input_size"], [224, 224], "input_size")
require_eq(config["training"]["batch_size"], 64, "batch_size")
require_eq(config["training"]["epochs"], 10, "epochs")
require_eq(config["training"]["optimizer"], "AdamW", "optimizer")
require_eq(config["training"]["learning_rate"], 0.0001, "learning_rate")
require_eq(config["training"]["weight_decay"], 0.01, "weight_decay")
require_eq(config["training"]["loss"], "CrossEntropyLoss", "loss")
require_eq(config["training"]["scheduler"], "CosineAnnealingLR", "scheduler")
require_eq(config["training"]["mixed_precision"], True, "mixed_precision")
require_eq(config["runtime"]["device_policy"], "cuda_if_available_else_cpu", "device_policy")

print("EXP-001 configuration verified successfully.")

In [ ]:
# 7. Verify manifest
import hashlib

manifest_path = Path(config["dataset"]["split_manifest"])
expected_sha = config["dataset"]["split_manifest_sha256"]

if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found at {manifest_path}")

with open(manifest_path, "rb") as f:
    computed_sha = hashlib.sha256(f.read()).hexdigest()

if computed_sha != expected_sha:
    raise RuntimeError(f"STOP. Manifest SHA-256 mismatch!\nExpected: {expected_sha}\nGot: {computed_sha}")

print(f"Manifest SHA-256 verified successfully: {computed_sha}")

In [ ]:
# 8. Verify split counts
import csv

split_counts = {"train": {"REAL": 0, "FAKE": 0}, 
                "val": {"REAL": 0, "FAKE": 0}, 
                "test": {"REAL": 0, "FAKE": 0}}

with open(manifest_path, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        split = row["split"]
        label = row["label"].upper()
        if split in split_counts and label in split_counts[split]:
            split_counts[split][label] += 1

print("Split distributions:")
for split, counts in split_counts.items():
    total = sum(counts.values())
    print(f"{split}: {total} total (FAKE {counts['FAKE']}, REAL {counts['REAL']})")

def require_count(actual, expected, name):
    if actual != expected:
        raise RuntimeError(f"STOP. Split mismatch for {name}: expected {expected}, got {actual}")

require_count(sum(split_counts["train"].values()), 98000, "train total")
require_count(split_counts["train"]["FAKE"], 49000, "train FAKE")
require_count(split_counts["train"]["REAL"], 49000, "train REAL")

require_count(sum(split_counts["val"].values()), 21000, "val total")
require_count(split_counts["val"]["FAKE"], 10500, "val FAKE")
require_count(split_counts["val"]["REAL"], 10500, "val REAL")

require_count(sum(split_counts["test"].values()), 21000, "test total")
require_count(split_counts["test"]["FAKE"], 10500, "test FAKE")
require_count(split_counts["test"]["REAL"], 10500, "test REAL")

print("Split counts verified perfectly.")

In [ ]:
# 9. Google Drive Dataset
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab or google.colab unavailable. Assuming local execution for preflight testing.")

DRIVE_ZIP_PATH = "/content/drive/MyDrive/DeepVerify/datasets/140k-real-and-fake-faces.zip"

if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(f"STOP. Verified dataset ZIP not found at {DRIVE_ZIP_PATH}")

print(f"Dataset ZIP found at {DRIVE_ZIP_PATH}")

In [ ]:
# 10. Verify ZIP SHA-256
print(f"Calculating SHA-256 for {DRIVE_ZIP_PATH}. This may take a few minutes...")
with open(DRIVE_ZIP_PATH, "rb") as f:
    zip_sha = hashlib.sha256()
    for chunk in iter(lambda: f.read(4096 * 1024), b""):
        zip_sha.update(chunk)
zip_computed = zip_sha.hexdigest()

expected_zip_sha = "ddaec55d781628e9f356543aa05c4946c3e6670b9232998b538579332ce6929f"
if zip_computed != expected_zip_sha:
    raise RuntimeError(f"STOP. Dataset ZIP SHA-256 mismatch!\nExpected: {expected_zip_sha}\nGot: {zip_computed}")

print(f"Dataset ZIP SHA-256 verified perfectly: {zip_computed}")

In [ ]:
# 11. Safe ZIP Extraction & Deterministic Normalization
import zipfile
import shutil
import csv
from pathlib import Path
import collections

extract_dir = "/content/DeepVerify/data/raw/"
extract_dir_path = Path(extract_dir).resolve()
expected_140k_dir = extract_dir_path / "140k"

if expected_140k_dir.exists():
    print("Previous canonical extraction detected. Removing to ensure clean state...")
    shutil.rmtree(expected_140k_dir)

print(f"Extracting ZIP to {extract_dir_path}...")
with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
    for member in z.infolist():
        member_path = Path(member.filename)
        if member_path.is_absolute() or ".." in member_path.parts:
            raise RuntimeError(f"STOP. Malicious path detected in ZIP: {member.filename}")
        target_path = (extract_dir_path / member.filename).resolve()
        if extract_dir_path not in target_path.parents and extract_dir_path != target_path:
            raise RuntimeError(f"STOP. Path traversal attempt: {member.filename}")
    z.extractall(path=extract_dir_path)
print("Raw extraction complete.")

# 11b. Normalization
print("Normalizing extracted files to canonical layout...")
extracted_files = list(extract_dir_path.rglob("*.jpg"))
basename_to_src = {}
for f in extracted_files:
    if f.name in basename_to_src:
        raise RuntimeError(f"STOP. Ambiguous filename detected in extracted ZIP: {f.name}")
    basename_to_src[f.name] = f

(expected_140k_dir / "real").mkdir(parents=True, exist_ok=True)
(expected_140k_dir / "fake").mkdir(parents=True, exist_ok=True)

manifest_path = Path(config["dataset"]["split_manifest"])
missing = []
with open(manifest_path, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        target_filepath = Path("/content/DeepVerify") / row["filepath"]
        basename = target_filepath.name
        if basename not in basename_to_src:
            missing.append(basename)
            continue
        # Use shutil.move to avoid duplicating storage (140k images)
        shutil.move(str(basename_to_src[basename]), str(target_filepath))

if missing:
    raise RuntimeError(f"STOP. {len(missing)} manifest files missing from ZIP. Examples: {missing[:5]}")

# Clean up the leftover messy directories from the ZIP (e.g., real_vs_fake)
for child in extract_dir_path.iterdir():
    if child.is_dir() and child.name != "140k":
        shutil.rmtree(child)

print("Deterministic normalization complete. Storage efficiency preserved via moves.")


In [ ]:
# 12. Verify Extracted Dataset
raw_140k_path = Path("/content/DeepVerify/data/raw/140k").resolve()
if not raw_140k_path.exists():
    raise RuntimeError(f"STOP. Expected extraction path does not exist: {raw_140k_path}")

fake_dir = raw_140k_path / "fake"
real_dir = raw_140k_path / "real"

if not fake_dir.exists():
    raise RuntimeError("STOP. 'fake' directory missing.")
if not real_dir.exists():
    raise RuntimeError("STOP. 'real' directory missing.")

fake_files = list(fake_dir.glob("*"))
real_files = list(real_dir.glob("*"))

if len(fake_files) != 70000:
    raise RuntimeError(f"STOP. Expected exactly 70,000 files in 'fake', got {len(fake_files)}")
if len(real_files) != 70000:
    raise RuntimeError(f"STOP. Expected exactly 70,000 files in 'real', got {len(real_files)}")

total_files = len(fake_files) + len(real_files)
if total_files != 140000:
    raise RuntimeError(f"STOP. Expected exactly 140,000 total files, got {total_files}")

print("Dataset filesystem integrity verified: 140,000 canonical files found.")

# Verify all manifest paths exist and belong to the correct location
missing_files = []
row_count = 0
with open(manifest_path, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        row_count += 1
        filepath = (Path("/content/DeepVerify") / row["filepath"]).resolve()
        
        if not filepath.exists():
            missing_files.append(str(filepath))
        
        # Verify paths belong exclusively to the data/raw/140k directory tree
        if raw_140k_path not in filepath.parents:
            raise RuntimeError(f"STOP. Manifest path does not belong to data/raw/140k: {filepath}")

if row_count != 140000:
    raise RuntimeError(f"STOP. Manifest row count is {row_count}, expected 140,000.")

if missing_files:
    raise RuntimeError(f"STOP. {len(missing_files)} files from the manifest are missing on disk.\nExamples: {missing_files[:5]}")

print("Manifest filepath alignment perfectly verified. All 140,000 paths exist on disk at their exact canonical locations.")


In [ ]:
# 13. Load tiny samples from real dataset
from deepverify_ml.data.dataset import build_dataloader_from_config
from PIL import Image

train_loader = build_dataloader_from_config(config, split="train")
val_loader = build_dataloader_from_config(config, split="val")

# Do NOT instantiate the test DataLoader.

train_batch, train_labels = next(iter(train_loader))
val_batch, val_labels = next(iter(val_loader))

print(f"Train batch shape: {train_batch.shape}, dtype: {train_batch.dtype}")
print(f"Train labels shape: {train_labels.shape}, dtype: {train_labels.dtype}")

if list(train_batch.shape[1:]) != [3, 224, 224]:
    raise RuntimeError(f"STOP. Incorrect train batch shape: {train_batch.shape}")
if train_batch.dtype != torch.float32:
    raise RuntimeError(f"STOP. Incorrect train batch dtype: {train_batch.dtype}")

print("Preprocessing and dataloading verified successfully on REAL extracted dataset.")

In [ ]:
# 14. GPU model forward pass
from deepverify_ml.models.efficientnet import build_model_from_config

model = build_model_from_config(config)
model = model.cuda()

# Tiny batch forward pass
test_batch = train_batch[:2].cuda()
with torch.no_grad():
    logits = model(test_batch)
    
print(f"Forward pass output shape: {logits.shape}")

if list(logits.shape) != [2, 2]:
    raise RuntimeError(f"STOP. Incorrect forward pass logits shape: {logits.shape}")

print("Outputs are raw logits.")

In [ ]:
# 15. Tiny GPU training smoke test
import tempfile
from deepverify_ml.training.trainer import DeepVerifyTrainer, build_optimizer, build_scheduler, build_loss, set_seed
from torch.amp import GradScaler

set_seed(config["experiment"]["seed"])

# Use a temporary directory for artifacts to ensure artifact safety
with tempfile.TemporaryDirectory() as tmpdir:
    trainer = DeepVerifyTrainer(config, run_dir=tmpdir)
    
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config)
    criterion = build_loss(config).cuda()
    scaler = GradScaler("cuda", enabled=trainer.use_amp)
    
    print("Running 1 tiny synthetic batch iteration for smoke testing on real images...")
    
    model.train()
    optimizer.zero_grad()
    
    with torch.autocast(device_type="cuda", enabled=trainer.use_amp):
        outputs = model(test_batch)
        test_labels_batch = train_labels[:2].cuda()
        loss = criterion(outputs, test_labels_batch)
        
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    scheduler.step()
    
    print("Tiny GPU training smoke test passed! (Model, CUDA, loss, backward, optimizer, AMP, scheduler all work)")

In [ ]:
# 16. STOP
print("==================================================")
print("EXP-001 PREFLIGHT COMPLETE — REAL TRAINING NOT STARTED")
print("==================================================")